
# Lab: Fisher Discriminant Analysis with Iris, Wine and Digits

This lab is based on the class PPT on Fisher Discriminant Analysis.

By the end of the lab, students should be able to:

1. Explain why feature importance depends on the task.
2. Compute a Fisher-style score for every feature.
3. Interpret the 4-panel Iris distribution plot and the red Fisher scores.
4. Compare PCA and Fisher/LDA projections visually.
5. Map dataset target codes to class names for Iris, Wine and Digits.
6. Select digits 5, 6 and 8, then project them with PCA and LDA.

Datasets used: Iris, Wine and Digits from scikit-learn.


In [ ]:
# This cell imports all libraries needed for the lab. Each line says what it does and why we need it.

import numpy as np  # Imports NumPy for arrays, numeric operations and formulas used in Fisher scoring.
import pandas as pd  # Imports pandas so we can show feature scores and mappings as readable tables.
import matplotlib.pyplot as plt  # Imports Matplotlib so we can draw distribution and projection plots.
from sklearn.datasets import load_iris, load_wine, load_digits  # Imports built-in datasets used in this lab.
from sklearn.preprocessing import StandardScaler  # Imports scaling because PCA and LDA work better when feature scales are comparable.
from sklearn.decomposition import PCA  # Imports PCA so we can project high-dimensional data to 2D using overall variance.
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis  # Imports LDA to create Fisher-style projections using class labels.


## 1. Fisher score idea

For a single feature, Fisher scoring asks:

- Are class means far apart?
- Is the spread inside each class small?

For two classes, the classroom idea is:

$$
F_d(a,b)=\frac{(\mu_{d,a}-\mu_{d,b})^2}{s_{d,a}^2+s_{d,b}^2}
$$

For more than two classes, we add the pairwise scores across class pairs and weight each pair by class probabilities.

In this lab, higher Fisher score means the feature is better for separating classes.


In [ ]:
# This cell loads Iris data and prints the target-code mapping.

iris = load_iris() 
X_iris = iris.____  # TODO: Replace ____ with the attribute that stores the feature matrix.
y_iris = iris.____  # TODO: Replace ____ with the attribute that stores the target labels.
iris_feature_names = iris.____  # TODO: Replace ____ with the attribute that stores feature names.
iris_target_names = iris.____  # TODO: Replace ____ with the attribute that stores class names.
iris_df = pd.DataFrame(X_iris, columns=iris_feature_names)  
iris_df["target_code"] = y_iris  
iris_df["target_name"] = iris_target_names[y_iris]  # Adds readable class names so the target mapping becomes clear.
iris_mapping = pd.DataFrame({"target_code": [0, 1, 2], "target_name": iris_target_names})  # Builds the Iris code-to-name mapping table.
print("Iris X shape:", X_iris.shape) 
print("Iris target-code mapping:")  
display(iris_mapping)  # Displays 0 as setosa, 1 as versicolor and 2 as virginica.
print("Iris class counts:") 
display(iris_df["target_name"].value_counts().reset_index())  
display(iris_df.head())

In [ ]:
# This cell defines a reusable Fisher-score function for multi-class data.

def fisher_scores_all_classes(X, y, feature_names):  # Defines a function so the same Fisher calculation can be reused for Iris and Wine.
    X = np.asarray(X)  # Converts input features to a NumPy array so slicing and numeric calculations are reliable.
    y = np.asarray(y)  # Converts labels to a NumPy array so Boolean masks work correctly.
    classes = np.unique(y)  # Finds unique class labels because Fisher score compares class pairs.
    total_n = len(y)  # Stores the total number of rows so class probabilities can be calculated.
    scores = []  # Creates an empty list to collect one Fisher score for each feature.
    for feature_index, feature_name in enumerate(feature_names): 
        feature_score = 0.0  
        for i, class_a in enumerate(classes):  
            for class_b in classes[i + 1:]:  
                values_a = X[y == class_a, feature_index]  # Gets this feature's values for class a.
                values_b = X[y == class_b, feature_index]  # Gets this feature's values for class b.
                mean_a = values_a.____()  # TODO: Replace ____ with the method that computes the average.
                mean_b = values_b.____()  # TODO: Replace ____ with the method that computes the average.
                var_a = values_a.var(ddof=____)  # TODO: Use ddof=1 for sample variance.
                var_b = values_b.var(ddof=____)  # TODO: Use ddof=1 for sample variance.
                prob_a = len(values_a) / total_n  # Computes class-a probability for weighting.
                prob_b = len(values_b) / total_n  # Computes class-b probability for weighting.
                denominator = var_a + var_b  # Adds within-class variances because Fisher penalizes spread.
                pair_score = ((mean_a - mean_b) ** 2) / denominator  # Computes separation divided by spread.
                feature_score = feature_score + (prob_a * prob_b * pair_score)  # Adds weighted pair score to this feature's total.
        scores.append({"feature": feature_name, "fisher_score": feature_score})  # Saves the feature name and final score.
    score_df = pd.DataFrame(scores)  # Converts the list of dictionaries into a table.
    score_df = score_df.sort_values("fisher_score", ascending=____)  # TODO: Use False so highest scores appear first.
    score_df = score_df.reset_index(drop=True)  # Resets row numbers after sorting.
return score_df  # Returns the ranked Fisher-score table.

In [ ]:
# This cell calculates and ranks Iris features using Fisher scores.

iris_scores = ____(X_iris, y_iris, iris_feature_names)  # TODO: Call the Fisher-score function defined above.
print("Iris features ranked by Fisher score:") 
display(iris_scores)

In [ ]:
# This cell creates the important 4-panel Iris distribution plot with Fisher scores shown in red.


def normal_pdf(x_values, mean, std):  # Defines a normal curve function so each class distribution can be drawn smoothly.
    coefficient = 1 / (std * np.sqrt(2 * np.pi))  # Computes the front part of the normal density formula.
    exponent = -0.5 * ((x_values - mean) / std) ** 2  # Computes the exponent part that controls the bell shape.
    density = coefficient * np.exp(exponent)  # Computes the final density values for the curve.
    return density  # Returns the curve heights so Matplotlib can plot them.

fig, axes = plt.subplots(____, ____, figsize=(12, 7))  # TODO: Create a 2 by 2 grid because Iris has 4 features.
axes = axes.ravel()  # Flattens the axes array so we can loop over it using one index.
for feature_index, feature_name in enumerate(iris_feature_names):  # Loops through all Iris features so each gets one panel.
    ax = axes[feature_index]  # Selects the subplot for the current feature.
    x_min = X_iris[:, feature_index].min() - 0.5  # Creates a left plotting limit below the smallest value.
    x_max = X_iris[:, feature_index].max() + 0.5  # Creates a right plotting limit above the largest value.
    x_grid = np.linspace(x_min, x_max, 300)  # Creates many x positions so the curves look smooth.
    for class_id, class_name in enumerate(iris_target_names):  # Loops through each Iris class.
        class_values = X_iris[y_iris == class_id, feature_index]  # Selects values for the current class and feature.
        class_mean = class_values.mean()  # Computes the mean used as the center of the class curve.
        class_std = class_values.std(ddof=1)  # Computes the standard deviation used as the width of the class curve.
        y_density = normal_pdf(x_grid, class_mean, class_std)  # Computes the bell curve values.
        ax.plot(x_grid, y_density, label=class_name)  # Draws the class distribution curve.
    score_value = iris_scores.loc[iris_scores["feature"] == feature_name, "fisher_score"].iloc[0]  # Gets the Fisher score for this feature.
    ax.text(0.66, 0.88, f"Fisher: {score_value:.4f}", transform=ax.transAxes, color=____, fontsize=12, fontweight="bold")  # TODO: Use "red" for the Fisher score.
    ax.set_title(feature_name)  # Uses the feature name as the panel title.
    ax.set_xlabel("feature value")  # Labels the x-axis.
    ax.set_ylabel("density")  # Labels the y-axis.
    ax.grid(True, alpha=0.5)  # Adds a light grid.
axes[0].legend()  # Shows the class-color mapping once.
plt.tight_layout()  # Adjusts spacing so labels do not overlap.
plt.show()  # Displays the completed 4-panel plot.

In [ ]:
# This cell loads Wine data, prints mappings and ranks Wine features by Fisher score.

wine = load_wine()
X_wine = wine.____  # TODO: Replace ____ with the attribute that stores the feature matrix.
y_wine = wine.____  # TODO: Replace ____ with the attribute that stores the target labels.
wine_feature_names = wine.____  # TODO: Replace ____ with the attribute that stores feature names.
wine_target_names = wine.____  # TODO: Replace ____ with the attribute that stores target names.
wine_df = pd.DataFrame(X_wine, columns=wine_feature_names)  # Creates a DataFrame so Wine values are easier to inspect.
wine_df["target_code"] = y_wine  # Adds numeric target codes.
wine_df["target_name"] = wine_target_names[y_wine]  # Adds readable target names.
wine_target_mapping = pd.DataFrame({"target_code": [0, 1, 2], "target_name": wine_target_names})  # Creates the Wine target-code mapping table.
wine_feature_mapping = pd.DataFrame({"feature_index": range(len(wine_feature_names)), "feature_name": wine_feature_names})  # Creates the Wine feature-index mapping table.
wine_scores = ____(X_wine, y_wine, wine_feature_names)  # TODO: Call the Fisher-score function.
print("Wine X shape:", X_wine.shape)  # Prints rows and columns.
print("Wine target-code mapping:")  # Prints a heading before the target mapping table.
display(wine_target_mapping)  # Displays Wine target-code mapping.
print("Wine feature-index mapping:")  # Prints a heading before the feature-index mapping table.
display(wine_feature_mapping)  # Displays all 13 Wine feature indices and names.
print("Wine class counts:")  # Prints a heading before class counts.
display(wine_df["target_name"].value_counts().reset_index())  # Shows class counts.
print("Top 5 Wine features by Fisher score:")  # Prints a heading before top-five feature scores.
display(wine_scores.head(____))  # TODO: Display the top 5 rows.

In [ ]:
# This cell creates the important 4-panel Wine distribution plot with Fisher scores shown in red.
# It closely follows the same structure as the Iris plotting cell.

top_wine_features = wine_scores_______["feature"].tolist()  # Selects the top 4 Wine features based on Fisher score.

fig, axes = ________________  # TODO: Creates a 2 by 2 grid because we want 4 important Wine panels.
axes = ________________  # TODO: Flattens the axes array so we can loop over it with one index.

for feature_index, feature_name in enumerate(___________):  # TODO: Loop through the selected top 4 Wine features.
    ax = axes[feature_index]  # Selects the subplot for the current feature.
    column_index = _____(_____________).index(feature_name)  # TODO: Finds the original Wine column index for the feature name.
    ______________________________  # TODO: Creates a left plotting limit slightly below the smallest value.
    ______________________________  # TODO: Creates a right plotting limit slightly above the largest value.
    ________________________________  # TODO: Creates many x positions so the distribution curves look smooth.

    for class_id, class_name in enumerate(wine_target_names):  # Loops through each Wine class to draw one curve per class.
        class_values = _____________________________  # Selects values for the current class and feature.
        class_mean = _____________________  # Computes the mean used as the center of the class curve.
        class_std = _________________  # Computes the standard deviation used as the width of the class curve.
        y_density = _____________________  # Computes the bell curve values for this class.
        _________________________________  # Draws the class distribution so overlap and separation can be seen.

    score_value = wine_scores.loc[wine_scores["feature"] == feature_name, "fisher_score"].iloc[0]  # Gets the Fisher score for the current feature.
    _____________________  # Writes the Fisher score in red.
    _______________________  # Uses the feature name as the panel title.
    _________________  # Labels the x-axis.
    _________________  # Labels the y-axis.
    _____________________  # Adds a light grid.

axes[0].legend()  
plt.tight_layout()
plt.show()  

In [ ]:
# This cell compares PCA and Fisher/LDA projection on the Iris dataset.

iris_scaler = ____()  # TODO: Create a StandardScaler object.
X_iris_scaled = iris_scaler.____(X_iris)  # TODO: Use fit_transform to scale Iris features.
iris_pca = ____(n_components=2, random_state=42)  # TODO: Create a PCA model with 2 components.
X_iris_pca = iris_pca.____(X_iris_scaled)  # TODO: Fit PCA and transform the scaled Iris data.
iris_lda = ____(n_components=2)  # TODO: Create a LinearDiscriminantAnalysis model with 2 components.
X_iris_lda = iris_lda.____(X_iris_scaled, y_iris)  # TODO: Fit LDA using X and y, then transform.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))  # Creates two side-by-side plots.
for class_id, class_name in enumerate(iris_target_names):  # Loops through each Iris class for PCA.
    axes[0].scatter(X_iris_pca[y_iris == class_id, 0], X_iris_pca[y_iris == class_id, 1], label=class_name, alpha=0.75)  # Plots PCA points for one class.
axes[0].set_title("Iris PCA projection")  # Titles the PCA plot.
axes[0].set_xlabel("PC1")  # Labels the first PCA axis.
axes[0].set_ylabel("PC2")  # Labels the second PCA axis.
axes[0].grid(True, alpha=0.5)  # Adds a grid.
for class_id, class_name in enumerate(iris_target_names):  # Loops through each Iris class for LDA.
    axes[1].scatter(X_iris_lda[y_iris == class_id, 0], X_iris_lda[y_iris == class_id, 1], label=class_name, alpha=0.75)  # Plots LDA points for one class.
axes[1].set_title("Iris Fisher/LDA projection")  # Titles the LDA plot.
axes[1].set_xlabel("LD1")  # Labels the first discriminant axis.
axes[1].set_ylabel("LD2")  # Labels the second discriminant axis.
axes[1].legend()  # Shows class-color mapping.
axes[1].grid(True, alpha=0.5)  # Adds a grid.
plt.tight_layout()  # Adjusts spacing.
plt.show()  # Displays the comparison figure.
print("Iris PCA explained variance ratio:", iris_pca.____)  # TODO: Print explained_variance_ratio_.
print("Note: versicolor and virginica may still overlap because LDA improves linear separation but does not guarantee perfect separation.")  # Helps students interpret the plot.

In [ ]:
# This cell compares PCA and Fisher/LDA projection on the Wine dataset.

wine_scaler = ____()  # TODO: Create a StandardScaler object.
X_wine_scaled = wine_scaler.____(X_wine)  # TODO: Use fit_transform to scale Wine features.
wine_pca = ____(n_components=2, random_state=42)  # TODO: Create a PCA model with 2 components.
X_wine_pca = wine_pca.____(X_wine_scaled)  # TODO: Fit PCA and transform scaled Wine data.
wine_lda = ____(n_components=2)  # TODO: Create a LinearDiscriminantAnalysis model with 2 components.
X_wine_lda = wine_lda.____(X_wine_scaled, y_wine)  # TODO: Fit LDA using X and y, then transform.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))  
for class_id, class_name in enumerate(wine_target_names): 
    axes[0].scatter(X_wine_pca[y_wine == class_id, 0], X_wine_pca[y_wine == class_id, 1], label=class_name, alpha=0.75)  
axes[0].set_title("Wine PCA projection")  
axes[0].set_xlabel("PC1")  
axes[0].set_ylabel("PC2") 
axes[0].grid(True, alpha=0.5)  
for class_id, class_name in enumerate(wine_target_names):  
    axes[1].scatter(X_wine_lda[y_wine == class_id, 0], X_wine_lda[y_wine == class_id, 1], label=class_name, alpha=0.75)  
axes[1].set_title("Wine Fisher/LDA projection")  
axes[1].set_xlabel("LD1")  
axes[1].set_ylabel("LD2")  
axes[1].legend()  
axes[1].grid(True, alpha=0.5)  
plt.tight_layout()  
plt.show()  
print("Wine PCA explained variance ratio:", wine_pca.____)  # TODO: Print explained_variance_ratio_.
print("Note: Wine often shows LDA separation more clearly because LDA uses labels while PCA does not.") 

In [ ]:
# This cell loads the Digits dataset, keeps digits 5, 6 and 8 and shows example images.

digits = load_digits()  # Loads the Digits dataset because it gives small 8 by 8 handwritten digit images.
X_digits = digits.____  # TODO: Replace ____ with the attribute that stores flattened 64-feature images.
y_digits = digits.____  # TODO: Replace ____ with the attribute that stores digit labels.
digit_images = digits.____  # TODO: Replace ____ with the attribute that stores 8 by 8 image arrays.
selected_digits = np.array([____, ____, ____])  # TODO: Select digits 5, 6 and 8.
digit_mask = np.isin(y_digits, selected_digits)  # Creates a Boolean mask for selected digits.
X_digits_568 = X_digits[digit_mask]  # Keeps only selected digit feature rows.
y_digits_568 = y_digits[digit_mask]  # Keeps only selected digit labels.
images_568 = digit_images[digit_mask]  # Keeps only selected digit images.
digits_mapping = pd.DataFrame({"target_code": selected_digits, "target_name": ["digit_5", "digit_6", "digit_8"]})  # Creates a mapping table for selected digits.
print("Selected Digits X shape:", X_digits_568.shape) 
print("Selected digit mapping:")  
display(digits_mapping)  
print("Selected digit class counts:")  
display(pd.Series(y_digits_568).value_counts().sort_index().reset_index(name="count"))  # Displays counts for selected digits.
fig, axes = plt.subplots(1, 3, figsize=(7, 2.5))  # Creates one row of three panels.
for plot_index, digit_value in enumerate(selected_digits):  
    first_index = np.where(y_digits_568 == digit_value)[0][0]  # Finds first sample index for the current digit.
    axes[plot_index].imshow(images_568[first_index], cmap="gray_r")  # Displays the 8 by 8 image.
    axes[plot_index].set_title(f"Digit {digit_value}")  
    axes[plot_index].axis("off")  
plt.tight_layout()  
plt.show() 

In [ ]:
# This cell samples digits 5, 6 and 8, then compares PCA and Fisher/LDA projection.

rng = np.random.default_rng(42)  # Creates a random generator so the sample is reproducible.
samples_per_digit = ____  # TODO: Use 70 examples per digit.
chosen_indices = []  # Creates an empty list to collect selected row indices.
for digit_value in selected_digits:  # Loops through each selected digit.
    class_indices = np.where(y_digits_568 == digit_value)[0]  # Finds all row indices for the current digit.
    sampled_indices = rng.choice(class_indices, size=samples_per_digit, replace=False)  # Randomly samples balanced rows.
    chosen_indices.extend(sampled_indices)  # Adds sampled row indices to the full list.
chosen_indices = np.array(chosen_indices) 
X_digits_sample = X_digits_568[chosen_indices] 
y_digits_sample = y_digits_568[chosen_indices]  
digits_scaler = ____()  # TODO: Create a StandardScaler object.
X_digits_sample_scaled = digits_scaler.____(X_digits_sample)  # TODO: Use fit_transform to scale sampled digits.
digits_pca = ____(n_components=2, random_state=42)  # TODO: Create a PCA model with 2 components.
X_digits_pca = digits_pca.____(X_digits_sample_scaled)  # TODO: Fit PCA and transform sampled digits.
digits_lda = ____(n_components=2)  # TODO: Create a LinearDiscriminantAnalysis model with 2 components.
X_digits_lda = digits_lda.____(X_digits_sample_scaled, y_digits_sample)  # TODO: Fit LDA using X and y, then transform.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))  
for digit_value in selected_digits:  # Loops through each digit for PCA.
    axes[0].scatter(X_digits_pca[y_digits_sample == digit_value, 0], X_digits_pca[y_digits_sample == digit_value, 1], label=f"digit {digit_value}", alpha=0.75)  
axes[0].set_title("Digits 5, 6, 8 PCA projection")  
axes[0].set_xlabel("PC1")  
axes[0].set_ylabel("PC2") 
axes[0].grid(True, alpha=0.5) 
for digit_value in selected_digits:  # Loops through each digit for LDA.
    axes[1].scatter(X_digits_lda[y_digits_sample == digit_value, 0], X_digits_lda[y_digits_sample == digit_value, 1], label=f"digit {digit_value}", alpha=0.75)  
axes[1].set_title("Digits 5, 6, 8 Fisher/LDA projection")  
axes[1].set_xlabel("LD1")  
axes[1].set_ylabel("LD2") 
axes[1].legend()  
axes[1].grid(True, alpha=0.5)  
plt.tight_layout()  
plt.show()  
print("Digits PCA explained variance ratio:", digits_pca.____)  # TODO: Print explained_variance_ratio_.
print("Note: LDA uses digit labels, so it usually separates 5, 6 and 8 more directly than PCA.")  


## Exit questions

Answer these after running the lab.

1. In the Iris Fisher-score table, which feature has the highest score?

    Ans:

2. In the 4-panel Iris distribution plot, why do petal length and petal width get higher Fisher scores than sepal width?

3. What does PCA try to preserve when it projects Iris, Wine or Digits data to 2D?

4. What does Fisher/LDA try to improve when it projects data to 2D?

5. Why are versicolor and virginica not perfectly separated in the Iris LDA plot?

6. In the Wine PCA vs LDA plots, which projection separates the three Wine classes more clearly and why?

7. In the Digits section, what are the selected target labels and what do they mean?

8. Why do we scale Wine and Digits before PCA and LDA?



## Functions, arguments and attributes used

| Tool | Syntax used | What it does | Documentation |
|---|---|---|---|
| `load_iris()` | `iris = load_iris()` | Loads 150 Iris samples, 4 features and 3 class labels. | https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html |
| `load_wine()` | `wine = load_wine()` | Loads 178 Wine samples, 13 features and 3 class labels. | https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html |
| `load_digits()` | `digits = load_digits()` | Loads 8 by 8 handwritten digit images as 64-feature rows. | https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html |
| `.data` | `dataset.data` | Returns the feature matrix `X`. | https://scikit-learn.org/stable/datasets.html |
| `.target` | `dataset.target` | Returns numeric class labels `y`. | https://scikit-learn.org/stable/datasets.html |
| `.feature_names` | `dataset.feature_names` | Returns readable feature names when available. | https://scikit-learn.org/stable/datasets.html |
| `.target_names` | `dataset.target_names` | Returns readable class names when available. | https://scikit-learn.org/stable/datasets.html |
| `pd.DataFrame()` | `pd.DataFrame(data, columns=names)` | Builds a table from arrays. | https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html |
| `.value_counts()` | `series.value_counts()` | Counts how often each label appears. | https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html |
| `.sort_values()` | `df.sort_values('column', ascending=False)` | Sorts rows by a column. | https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html |
| `.reset_index()` | `df.reset_index(drop=True)` | Resets row numbering after sorting or counting. | https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reset_index.html |
| `np.asarray()` | `np.asarray(X)` | Converts input to a NumPy array. | https://numpy.org/doc/stable/reference/generated/numpy.asarray.html |
| `np.unique()` | `np.unique(y)` | Finds unique class labels. | https://numpy.org/doc/stable/reference/generated/numpy.unique.html |
| `.mean()` | `values.mean()` | Calculates the average value. | https://numpy.org/doc/stable/reference/generated/numpy.mean.html |
| `.var(ddof=1)` | `values.var(ddof=1)` | Calculates sample variance. `ddof=1` uses sample variance. | https://numpy.org/doc/stable/reference/generated/numpy.var.html |
| `.std(ddof=1)` | `values.std(ddof=1)` | Calculates sample standard deviation. | https://numpy.org/doc/stable/reference/generated/numpy.std.html |
| `np.linspace()` | `np.linspace(start, stop, num)` | Creates evenly spaced values for smooth curves. | https://numpy.org/doc/stable/reference/generated/numpy.linspace.html |
| `np.exp()` | `np.exp(values)` | Calculates exponential values for the normal-density formula. | https://numpy.org/doc/stable/reference/generated/numpy.exp.html |
| `np.isin()` | `np.isin(y, [5,6,8])` | Creates a mask for selected classes. | https://numpy.org/doc/stable/reference/generated/numpy.isin.html |
| `np.where()` | `np.where(condition)` | Finds positions where a condition is true. | https://numpy.org/doc/stable/reference/generated/numpy.where.html |
| `np.random.default_rng()` | `np.random.default_rng(42)` | Creates a reproducible random number generator. | https://numpy.org/doc/stable/reference/random/generator.html |
| `rng.choice()` | `rng.choice(values, size=n, replace=False)` | Samples items randomly. | https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.choice.html |
| `plt.subplots()` | `plt.subplots(rows, cols, figsize=(w,h))` | Creates one or more plotting areas. | https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html |
| `ax.plot()` | `ax.plot(x, y, label=name)` | Draws line curves for distributions. | https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.plot.html |
| `ax.scatter()` | `ax.scatter(x, y, label=name, alpha=0.75)` | Draws 2D projected points. | https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.scatter.html |
| `ax.text()` | `ax.text(x, y, text, transform=ax.transAxes)` | Writes Fisher scores inside plots. | https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.text.html |
| `ax.imshow()` | `ax.imshow(image, cmap='gray_r')` | Displays digit images. | https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html |
| `StandardScaler()` | `scaler = StandardScaler()` | Standardizes features to mean 0 and variance 1. | https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html |
| `.fit_transform()` | `model.fit_transform(X)` or `model.fit_transform(X, y)` | Fits a transformer and returns transformed data. | https://scikit-learn.org/stable/glossary.html#term-fit_transform |
| `PCA(n_components=2)` | `PCA(n_components=2, random_state=42)` | Projects data to 2 axes that preserve maximum overall variance. | https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html |
| `.explained_variance_ratio_` | `pca.explained_variance_ratio_` | Shows the fraction of variance kept by each PCA component. | https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html |
| `LinearDiscriminantAnalysis()` | `LinearDiscriminantAnalysis(n_components=2)` | Creates class-separating Fisher/LDA axes. | https://scikit-learn.org/stable/modules/generated/sklearn.discriminant_analysis.LinearDiscriminantAnalysis.html |



## Exit questions

Answer these after running the lab.

1. In the Iris Fisher-score table, which feature has the highest score?
Petal length has the highest Fisher score.

2. In the 4-panel Iris distribution plot, why do petal length and petal width get higher Fisher scores than sepal width?
The two features of petal length and petal width are more clearly separated among the three species, with little overlapping. There is considerable overlap of sepal width in terms of classes, hence poor between-class separability and low Fisher’s score.

3. What does PCA try to preserve when it projects Iris, Wine or Digits data to 2D?
PCA tries to preserve as much of the original data variance as possible. It finds directions of maximum variation and projects the data onto those directions in the Iris, Wine and Digits .

4. What does Fisher/LDA try to improve when it projects data to 2D?
LDA tries to maximize class separability by increasing the distance between class means while reducing the spread within each class.

5. Why are versicolor and virginica not perfectly separated in the Iris LDA plot?
Versicolor and virginica have similar feature values, especially in petal measurements, causing their distributions to overlap. Because of this overlap, even LDA cannot separate them perfectly.

6. In the Wine PCA vs LDA plots, which projection separates the three Wine classes more clearly and why?
LDA separates the three Wine classes more clearly because it uses class-label information to maximize discrimination between classes. PCA only focuses on preserving variance and ignores class labels.

7. In the Digits section, what are the selected target labels and what do they mean?
The selected target labels are 5, 6, and 8. They represent images of handwritten digits 5, 6, and 8 from the Digits dataset.

8. Why do we scale Wine and Digits before PCA and LDA?
Scaling ensures that all features contribute equally to the analysis. Without scaling, features with larger numerical ranges would dominate the computations, leading to biased PCA and LDA projections.
